In [9]:
train_df.groupby('polarity')['contextualModality'].value_counts()

polarity  contextualModality
NEG       ACTUAL                  61
          HYPOTHETICAL            24
          HEDGED                   4
          GENERIC                  1
POS       ACTUAL                1770
          HYPOTHETICAL           208
          GENERIC                164
          HEDGED                  95
Name: count, dtype: int64

In [1]:
import json
import csv
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig, Gemma3ForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import re
import time
import os
import random

k = 6 # for sampling EVENTs
CSV_PATH = "/home/jainv/dr_osb_lab/ChemoTask/entity_tags.csv"
BASE = "/home/jainv/nlp_models/qwen3_32B"
# BASE = "/home/jainv/nlp_models/gemma3_27B_it/"
# BASE = "/home/jainv/nlp_models/medgemma_27b_text_it/"
# ADAPTER = "/home/jainv/nlp_models/sact_gen_medgemma/"
# OUTPUT_PATH = "/home/jainv/dr_osb_lab/ChemoTask/generated_notes_chemotask.csv"
torch.set_float32_matmul_precision('high')


def is_invalid(note, allowed_events):
    """
    0: Valid
    1: Completely invalid
    2: Partially invalid (all events were not use), other checks passed
    """
    allowed_polarity = {'POS', 'NEG'}
    allowed_modality = {'HEDGED', 'GENERIC', 'HYPOTHETICAL', 'ACTUAL'}
    event_count = set()
    
    if not note.startswith('='):
        return 1

    events = list(re.finditer(r'<EVENT\s+([^>]+)>', note, flags=re.IGNORECASE))
    entities = list(re.finditer(r'<EVENT\s+[^>]+>\s*()\s*</EVENT>', note, flags=re.IGNORECASE))
    closing_tags = re.findall(r'</EVENT>', note, flags=re.IGNORECASE)
    
    if len(events) != len(closing_tags):
        return 1
    
    for match in events:
        attrs = match.group(1)
        polarity = re.search(r'polarity=([\'"]?)(\w+)\1', attrs)
        modality = re.search(r'modality=([\'"]?)(\w+)\1', attrs)
        if polarity and modality:
            polarity_val = polarity.group(2)
            modality_val = modality.group(2)
            if polarity_val.upper() not in allowed_polarity or modality_val.upper() not in allowed_modality:
                return 1
        else:
            return 1
        
    allowed_events = [i.lower() for i in allowed_events]
    for match in entities:
        if match.group(1).lower() not in allowed_events:
            return 1
        else:
            event_count.add(match.group(1).lower())
            
    if len(event_count) != len(allowed_events):
        return 2

    return 0

def get_notes(df, cancer_type, mod_dict):
    
    notes_list = []
    
#     for mod in mod_dict[cancer_type]:

    df2 = df[(df['cancer_type']==cancer_type) &
             (df['record_type']=='NOTE') &
            (df['entity_type']=='EVENT') &
            (df['contextualModality'].isin(mod_dict[cancer_type]))
            ]
        
    if df2.shape[0] >= 2:
        notes_list += df2['note_path'].sample(n=2).to_list()
    else:
        notes_list += list(df2['note_path'])

    return notes_list

def insert_tags(df, files):
    
    samples = []
    
    for fname in files[:1]:
        
        with open(fname, encoding='utf-8') as f:
            text = f.read()

        note_spans = df[df['note_path'] == fname]
        spans = note_spans[['span_start', 'span_end', 'polarity', 'contextualModality']].sort_values(by='span_start').values

        for start, end, pol, mod in reversed(spans):
            text = text[:end] + " </EVENT>" + text[end:]
            if isinstance(pol, str) and isinstance(mod, str):
                text = text[:start] + f"<EVENT polarity={pol} modality={mod}> " + text[start:]
        
        text = re.sub(r'\n\s*\n+', '\n\n', text)
        samples.append(text)

    return samples


# model_dir = MODEL_DIR

# config = AutoConfig.from_pretrained(BASE)
# config.use_sliding_window = False

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16,
# )

model = AutoModelForCausalLM.from_pretrained(
    BASE,
#     quantization_config=bnb_config,
    torch_dtype="auto",
    device_map="auto",
)

# model = PeftModel.from_pretrained(model, ADAPTER, device_map="auto")
# model.eval()

tokenizer = AutoTokenizer.from_pretrained(BASE)
# tokenizer.pad_token_id = tokenizer.eos_token_id

df = pd.read_csv(CSV_PATH)

Loading checkpoint shards:   0%|          | 0/14 [00:00<?, ?it/s]

In [25]:
sys_prompt = """You are a clinical‐note author and precise annotator specializing in oncology, with a focus on systemic anticancer therapy (SACT) timelines. Your task is to generate synthetic clinical notes that accurately reflect a provided SACT timeline—i.e., a list of [drug/regimen, temporal_relation, time_expression] triples—as narrative text in strict chronological order.

**A. Definition of <EVENT>:**  
An <EVENT> here refers **only** to any SACT drug/regimen which includes:
  - Traditional cytotoxic chemotherapy agents  
  - Endocrine therapy agents  
  - Targeted therapy agents  
  - Immunotherapy agents  
  - SACT regimen names  
  - Generic mention of the word "chemotherapy"  

**B. Tagging Schema:**  
  – Every SACT event mention must be *wrapped exactly* as `<EVENT polarity=POLARITY modality=MODALITY> … </EVENT>`
    • `POLARITY`: POS or NEG  
    • `MODALITY`: one of `{mod_vals}` only
  – Half of the events POS, half NEG.  
  – 0–2 generic “chemotherapy” mentions.  
  – Each of the four modalities appears at least once.
  - Only use/tag the drug or regimen from the provided list of triplets.
  - Always tag each and every SACT mention. No mention should be untagged.

**C. Timeline Injection Instructions:**  
You will also receive a structured “SACT timeline” as a list of triples: [drug/regimen, temporal_relation, time_expression], where `temporal_relation` ∈ [BEGINS-ON, ENDS-ON, CONTAINS, CONTAINS-1, BEFORE, NOTED-ON, OVERLAP] and `time_expression` is in `YYYY-MM-DD` or `YYYY-MM` format.

  - **BEGINS-ON**: therapy started on that date.  
  - **ENDS-ON**: therapy completed/discontinued on that date.  
  - **CONTAINS-1**: therapy was ongoing during the specified month (inverse of CONTAINS).
  
When writing the note:
  1. **Chronology**: Present each SACT event in strict order of the timeline.  
  2. **Phrasing**:  
     - BEGINS-ON → “Patient began <EVENT>…</EVENT> on MM/DD/YYYY.”  
     - ENDS-ON → “completed/discontinued <EVENT>…</EVENT> on MM/DD/YYYY.”  
     - CONTAINS-1 (consecutive months) → “continued <EVENT>…</EVENT> from MM/YYYY to MM/YYYY.”  
     - Single CONTAINS-1 → “continued <EVENT>…</EVENT> through MONTH YYYY.”  
     - BEFORE → “had previously completed <EVENT>…</EVENT> prior to MM/DD/YYYY.”  
     - NOTED-ON → “<EVENT>…</EVENT> was noted on MM/DD/YYYY.”  
     - OVERLAP → “<EVENT>…</EVENT> overlapped with [time period].”  
  3. **Principal Date**: Set the header’s Principal Date to a date on or after the latest timeline entry.  
  4. **No extra text** beyond the raw note structure, tagging, and timeline narrative.
  5. Use the exact same de‐identified header/footer and section styling as the provided samples (≈300–400 words total).  
  6. No extra headings, numbering, bullets, or commentary—only raw note text.  
  7. Set the header’s Principal Date to a date **on or after** the latest timeline entry.

*Strictly Follow all restrictions in their original form.*"""


usr_prompt = """Below are some de-identified sample clinical notes (≈300–400 words) that use only SACT drug/regimen <EVENT> tags. Study its format and how each agent/regimen is negated/affirmed and wrapped:

{sample_note}

---

**SACT Timeline for new note:**  
{timeline_list}

Now generate a new clinical note (≈300–350 words) for a patient with **{cancer_type}** cancer that:

- Strictly follows the <EVENT> tagging rules (polarity, modality, count constraints).  
- Accurately narrates the provided SACT Timeline in chronological order using the phrasing guidelines above.  
- Sets the Principal Date in the header to on or after the last timeline date.  
- Includes 0–2 generic “chemotherapy” mentions.  
- Does **not** tag any non-listed agent or stray tokens.  

**Always** write a complete note; you will be penalized if any instruction is violated."""

responses = []
mod_dict = {'breast':('HEDGED', 'GENERIC', 'HYPOTHETICAL', 'ACTUAL'), 
          'melanoma':('HEDGED', 'GENERIC', 'HYPOTHETICAL', 'ACTUAL'), 
          'ovarian':('HEDGED', 'GENERIC', 'HYPOTHETICAL', 'ACTUAL')
         }

df = df[df['split_type']=='train']
df = df[~((df['polarity']).isna() & (df['contextualModality']).isna())]

# with open('../allowed_events.json', 'r') as f:
#     events_list = json.load(f)

# if os.path.exists(OUTPUT_PATH):
#     pass
# else:
#     with open(OUTPUT_PATH, 'w') as f:
#         out = csv.writer(f)
#         out.writerow(["prompt", "cancer_type", "output", "generation_time"])

timelines_df = pd.read_csv('../generated_timelines.csv')[['cancer', 'output_processed']]

used_timelines = set()
p_start = time.time()
cancer_types = ['breast', 'melanoma', 'ovarian']
for cancer_type in cancer_types:
    print(f"\n\n{cancer_type}")
    sample_note = []
    
    chosen_notes = get_notes(df, cancer_type, mod_dict)
    timeline = timelines_df[(timelines_df['cancer']==cancer_type) & (~timelines_df['output_processed'].isin(used_timelines))]['output_processed'].sample(n=1).values[0]
    allowed_events = set()
    if isinstance(timeline, str):
        timeline = eval(timeline)

    for obj in timeline:
        allowed_events.add(obj[0])

    sample_note += insert_tags(df, chosen_notes)
    notes_string = '\n'.join(f'note {i+1}:\n{note}' for i, note in enumerate(sample_note))
    mod_vals_str = ', '. join(mod_dict[cancer_type])
#     regimen = random.choice(events_list[cancer_type]['regimens']) 
#     single_agents = random.sample(events_list[cancer_type]['single_agents'], k)
#     allowed_events = [regimen]+single_agents
    
    events_str = ', '.join(allowed_events)
    sys_prompt = sys_prompt.format(mod_vals=mod_vals_str, allowed_events=events_str)
    usr_prompt = usr_prompt.format(sample_note=notes_string, cancer_type=cancer_type, timeline_list=timeline)
    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": usr_prompt},
    ]
    
#     batch_output = []
#     batch_output_prompts = []
#     batch_output_time = []
#     batch_cancer_type = []
#     batch_invalid = []

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
#     model_inputs = tokenizer.apply_chat_template(
#         messages, add_generation_prompt=True, tokenize=True,
#         return_dict=True, return_tensors="pt"
#     ).to(model.device)
    
#     input_len = model_inputs["input_ids"].shape[-1]
    
    for i in range(1):
        
        s = time.time()
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=32768,
        )

#         generated_ids = model.generate(
#             **model_inputs,
#             max_new_tokens=5000,
#             do_sample=True,
#             temperature=0.6,
# #             top_p=0.9,
#             top_k=20,
#             pad_token_id = tokenizer.eos_token_id,
#         )
#         generated_ids = generated_ids[0][input_len:]
        
        
        end_time = (time.time()-s)/60
        
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

        try:
            index = len(output_ids) - output_ids[::-1].index(151668)
        except ValueError:
            index = 0

        
#         thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
        content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
#         content = tokenizer.decode(generated_ids, skip_special_tokens=True)
        content = re.split(r'---|self[\s\-_]?check|event summary', content, flags=re.IGNORECASE)[0].strip()
        content = content.replace('\\n','\n').replace('\\t','\t')
        if content.startswith("Report ID"):
            content = '===================================================================\n'+content

#         batch_output.append(content)
#         batch_output_prompts.append(usr_prompt)
#         batch_output_time.append(end_time)
#         batch_cancer_type.append(cancer_type)
#         batch_invalid.append(0 if content.startswith('=') else 1)
        
#         if i%5 == 0:
#             with open(OUTPUT_PATH, 'a', newline='') as f:
#                 out = csv.writer(f)
#                 out.writerows(list(zip(batch_output_prompts, batch_cancer_type, batch_output, batch_output_time, batch_invalid)))
#             print(f"Wrote {i} rows")
#             batch_output = []
#             batch_output_prompts = []
#             batch_output_time = []
#             batch_cancer_type = []

#     print(f"Took: {(time.time()-c_start)/3600:.2f} hrs for {i} notes\n\n")

        # For test run
#         print(thinking_content)
        code = is_invalid(content, allowed_events)
        if not code:
            print("\n\nMAIN CONTENT\n")
            print(content)
        else:
            print("Invalid:", code)
            print(content)
#         with open('../sample_data/event_modifiers3.txt', 'a') as f:
#             f.write(f"\n\n{cancer_type}\n\n")
#             f.write(f"\n\n{content}\n\n")
#             f.write("="*30)
            
    print(f"\n\nTook: {(time.time()-p_start)/3600:.2f} hrs")



breast
Invalid: 2
Report ID.....................91,ZXk9p2RnF7tq  
Patient ID....................ZXk9p2RnF7tq  
Patient Name..................Patient47  
Principal Date................20060710  
Record Type...................NOTE  
[Report de-identified (Limited dataset compliant) by De-ID v.6.24.5.1]  

Outpatient Prescriptions Prior to Visit  
 Medication Sig Dispense Refill  
  <EVENT polarity=POS modality=ACTUAL> AC </EVENT> (Adriamycin + Cyclophosphamide) regimen began on 01/10/2006.  
  <EVENT polarity=NEG modality=HEDGED> DOXORUBICIN </EVENT> therapy was not confirmed to continue through 02/01/2006.  
  <EVENT polarity=POS modality=GENERIC> CHEMOTHERAPY </EVENT> encompassed 02/01/2006 to 03/15/2006.  
  <EVENT polarity=NEG modality=ACTUAL> CYCLOPHOSPHAMIDE </EVENT> was discontinued on 04/05/2006.  
  <EVENT polarity=POS modality=HYPOTHETICAL> PACLITAXEL </EVENT> treatment may have overlapped with 04/05/2006 to 06/01/2006.  
  <EVENT polarity=POS modality=GENERIC> CHEMOTHERAPY <

In [2]:
import glob
import pandas as pd
import re

allowed_polarity = {'POS', 'NEG'}
allowed_modality = {'HEDGED', 'GENERIC', 'HYPOTHETICAL', 'ACTUAL'}

def is_invalid(note):
    
    if not note.startswith('='):
        return 1

    events = list(re.finditer(r'<EVENT\s+([^>]+)>', note, flags=re.IGNORECASE))
    closing_tags = re.findall(r'</EVENT>', note, flags=re.IGNORECASE)

    if len(events) != len(closing_tags):
        return 1

    for match in events:
        attrs = match.group(1)
        pol_m = re.search(r'polarity=([\'"]?)(\w+)\1', attrs, flags=re.IGNORECASE)
        mod_m = re.search(r'modality=([\'"]?)(\w+)\1', attrs, flags=re.IGNORECASE)
        if not (pol_m and mod_m):
            return 1

        pol_val = pol_m.group(2).upper()
        mod_val = mod_m.group(2).upper()
        if pol_val not in allowed_polarity or mod_val not in allowed_modality:
            return 1

    return 0


# df = pd.read_csv('../generated_notes_chemotask_{cancer_type}.csv')
# df = pd.read_csv('../generated_notes_chemotask_melanoma.csv')
# df = pd.read_csv('../generated_notes_chemotask_ovarian.csv')
dfs = []
for fn in glob.glob('../generated_notes_*.csv'):
    df = pd.read_csv(fn)
    if 'output' in df.columns:
        df = df.rename(columns={'output':'generated_note'})
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

# df['generated_note'] = df['generated_note'].apply(lambda x: x.replace('\\n','\n').replace('\\t','\t'))

# df['generated_note'] = df['generated_note'].apply(lambda x: re.sub(r',\s*==', '==', x))
# txt = '===================================================================\n'
# df['generated_note'] = df['generated_note'].apply(lambda x: txt+x if x.startswith("Report ID") else x)

# df['generated_note'] = df['generated_note'].apply(lambda x: re.split(r'---|self[\s\-_]?check|events? summary', x, flags=re.IGNORECASE)[0].strip())
# print(len(df[df['generated_note'].str.contains('--')]))
# df[df['generated_note'].str.contains('--')]

# print(df.iloc[908]['generated_note'])

# df['invalid'] = df['output'].apply(is_invalid)

# df[~df['generated_note'].str.startswith('=')]

# df.to_csv('../generated_notes_chemotask.csv', index=False)

print(df.shape[0], df['invalid'].sum())
print((df[df['invalid']==1]['invalid'].sum()/df.shape[0])*100)

7858 924
11.758717230847544
